In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, length, current_timestamp, from_utc_timestamp, row_number, udf, lit
from pyspark.sql.types import StringType, BooleanType
from pyspark.sql.window import Window
from pyspark.sql.functions import date_format, dayofweek

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_bronze"
SCHEMA_SILVER = "yelp_silver"
SCHEMA_GOLD = "yelp_gold"

def gerar_tabela_business_reviews():
    """
    Gera uma tabela integrada com dados de estabelecimentos e reviews.
    
    Retorna:
        DataFrame com colunas:
        - estabelecimento (nome)
        - food_category
        - review_id
        - stars (avaliação)
        - text (texto do review)
        - date (data do review)
        - dia_semana (nome do dia da semana)
        - dia_semana_numero (1=Domingo, 2=Segunda, ..., 7=Sábado)
        - estado (localização do estabelecimento)
    """
    
    print("Gerando tabela integrada Business + Reviews...")
    print("="*60)
    
    # Carrega tabelas silver
    df_business = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.business")
    df_review = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.review")
    
    print(f"Estabelecimentos: {df_business.count():,} registros")
    print(f"Reviews: {df_review.count():,} registros")
    
    # JOIN entre business e review
    df_integrado = df_review.join(
        df_business,
        df_review.business_id == df_business.business_id,
        'inner'
    )
    
    print(f"\nRegistros após JOIN: {df_integrado.count():,}")
    
    # Extrai dia da semana do review
    df_integrado = df_integrado.withColumn(
        'dia_semana_numero',
        dayofweek(col('date'))  # 1=Domingo, 2=Segunda, ..., 7=Sábado
    )
    
    # Converte número para nome do dia
    df_integrado = df_integrado.withColumn(
        'dia_semana',
        date_format(col('date'), 'EEEE')  # Nome completo do dia em inglês
    )
    
    # Seleciona e renomeia colunas
    df_resultado = df_integrado.select(
        df_business.name.alias('estabelecimento'),
        df_business.food_category,
        df_review.review_id,
        df_review.stars,
        df_review.date,
        col('dia_semana'),
        col('dia_semana_numero'),
        df_business.state.alias('estado')
    )
    
    print("\n✓ Tabela integrada gerada com sucesso!")
    print("="*60)
    
    return df_resultado

# Testa a função
df_tabela_integrada = gerar_tabela_business_reviews()

# Mostra amostra
print("\nAMOSTRA DA TABELA INTEGRADA (10 primeiros registros):")
print("-"*60)
display(df_tabela_integrada.limit(10))

# Estatísticas
print("\nDISTRIBUIÇÃO POR DIA DA SEMANA:")
display(
    df_tabela_integrada
    .groupBy('dia_semana', 'dia_semana_numero')
    .count()
    .orderBy('dia_semana_numero')
)

print("\nDISTRIBUIÇÃO POR ESTADO (Top 10):")
display(
    df_tabela_integrada
    .groupBy('estado')
    .count()
    .orderBy(col('count').desc())
    .limit(10)
)

print("\nDISTRIBUIÇÃO POR CATEGORIA DE COMIDA:")
display(
    df_tabela_integrada
    .groupBy('food_category')
    .count()
    .orderBy(col('count').desc())
)

In [0]:
# ========== SALVAR TABELA INTEGRADA COMO GOLD ==========

print("Salvando tabela integrada na camada Gold...")
print("="*60)

# Nome da tabela gold
table_name = f"{CATALOG}.{SCHEMA_GOLD}.gold_business_reviews_integrado"

# Adiciona metadados de processamento
df_tabela_final = df_tabela_integrada.withColumn(
    'data_processamento_silver',
    current_timestamp()
)

print(f"\nTotal de registros a salvar: {df_tabela_final.count():,}")
print(f"Campos: {len(df_tabela_final.columns)}")

# Salva a tabela
df_tabela_final.write.mode("overwrite").saveAsTable(table_name)

print(f"\n✓ Tabela gold salva com sucesso: {table_name}")
print("="*60)

# Verifica a tabela criada
print("\nVerificando tabela criada:")
df_verificacao = spark.table(table_name)

print(f"Total de registros na tabela: {df_verificacao.count():,}")
print(f"\nSchema da tabela:")
df_verificacao.printSchema()

# Mostra amostra
print("\nAmostra de 5 registros:")
display(df_verificacao.limit(5))

In [0]:
# ========== GRÁFICO: REVIEWS POR DIA DA SEMANA E CATEGORIA ==========

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("Gerando gráfico de reviews por dia da semana e categoria...")
print("="*60)

# Carrega a tabela integrada
df_integrado = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.gold_business_reviews_integrado")

# Agrupa por food_category e dia da semana
df_agrupado = df_integrado.groupBy('food_category', 'dia_semana', 'dia_semana_numero') \
    .count() \
    .orderBy('dia_semana_numero', 'food_category')

# Converte para pandas para facilitar a visualização
df_pandas = df_agrupado.toPandas()

print(f"Total de registros agrupados: {len(df_pandas)}")
print("\nAmostra dos dados:")
print(df_pandas.head(10))

# Ordena os dias da semana corretamente
dias_ordem = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
dias_pt = ['Domingo', 'Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado']

# Pivot para facilitar o gráfico
df_pivot = df_pandas.pivot(index='dia_semana', columns='food_category', values='count')
df_pivot = df_pivot.reindex(dias_ordem)
df_pivot = df_pivot.fillna(0)

print("\nDados pivotados:")
print(df_pivot)

# Configuração do gráfico
fig, ax = plt.subplots(figsize=(14, 8))

# Cores para cada categoria
cores = {
    'RESTAURANTE': '#FF6B6B',
    'BAR E BEBIDA': '#4ECDC4',
    'CAFE': '#FFE66D',
    'PADARIA': '#95E1D3',
    'OUTROS': '#A8E6CF'
}

# Posição das barras
x = np.arange(len(dias_pt))
width = 0.15  # Largura das barras

# Plota cada categoria
for i, categoria in enumerate(df_pivot.columns):
    offset = width * (i - len(df_pivot.columns)/2 + 0.5)
    cor = cores.get(categoria, '#CCCCCC')
    ax.bar(x + offset, df_pivot[categoria], width, label=categoria, color=cor, alpha=0.8)

# Configurações do gráfico
ax.set_xlabel('Dia da Semana', fontsize=12, fontweight='bold')
ax.set_ylabel('Quantidade de Reviews', fontsize=12, fontweight='bold')
ax.set_title('Distribuição de Reviews por Dia da Semana e Categoria de Comida', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(dias_pt, fontsize=10)
ax.legend(title='Categoria', fontsize=10, title_fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Adiciona valores nas barras (opcional - pode deixar o gráfico poluído)
# for container in ax.containers:
#     ax.bar_label(container, fmt='%.0f', fontsize=7, padding=2)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("✓ Gráfico gerado com sucesso!")
print("\nINSIGHTS:")
print("  • Domingo e Sábado tendem a ter mais reviews")
print("  • Restaurantes dominam em todos os dias da semana")
print("  • Quinta-feira tem o menor volume de reviews")
print("="*60)

In [0]:
# ========== ADICIONAR MACRORREGIÃO BASEADA NO CENSO DOS EUA ==========

from pyspark.sql.functions import when, col

print("Adicionando coluna de macrorregião às tabelas...")
print("="*80)

# Configuração
CATALOG = "workspace"
SCHEMA_GOLD = "yelp_gold"

# Mapeamento de estados para macrorregiões (Censo dos EUA)
MAPA_MACROREGIAO = {
    # NORDESTE
    'CT': 'Nordeste', 'ME': 'Nordeste', 'MA': 'Nordeste', 'NH': 'Nordeste', 
    'RI': 'Nordeste', 'VT': 'Nordeste',  # Nova Inglaterra
    'NJ': 'Nordeste', 'NY': 'Nordeste', 'PA': 'Nordeste',  # Médio Atlântico
    
    # CENTRO-OESTE
    'IL': 'Centro-Oeste', 'IN': 'Centro-Oeste', 'MI': 'Centro-Oeste', 
    'OH': 'Centro-Oeste', 'WI': 'Centro-Oeste',  # Leste Norte-Central
    'IA': 'Centro-Oeste', 'KS': 'Centro-Oeste', 'MN': 'Centro-Oeste', 
    'MO': 'Centro-Oeste', 'NE': 'Centro-Oeste', 'ND': 'Centro-Oeste', 
    'SD': 'Centro-Oeste',  # Oeste Norte-Central
    
    # SUL
    'DE': 'Sul', 'FL': 'Sul', 'GA': 'Sul', 'MD': 'Sul', 'NC': 'Sul', 
    'SC': 'Sul', 'VA': 'Sul', 'WV': 'Sul', 'DC': 'Sul',  # Atlântico Sul
    'AL': 'Sul', 'KY': 'Sul', 'MS': 'Sul', 'TN': 'Sul',  # Leste Sul-Central
    'AR': 'Sul', 'LA': 'Sul', 'OK': 'Sul', 'TX': 'Sul',  # Oeste Sul-Central
    
    # OESTE
    'AZ': 'Oeste', 'CO': 'Oeste', 'ID': 'Oeste', 'MT': 'Oeste', 
    'NV': 'Oeste', 'NM': 'Oeste', 'UT': 'Oeste', 'WY': 'Oeste',  # Montanha
    'AK': 'Oeste', 'CA': 'Oeste', 'HI': 'Oeste', 'OR': 'Oeste', 
    'WA': 'Oeste',  # Pacífico
    
    # CANADÁ (estados fora dos EUA)
    'AB': 'Canadá', 'BC': 'Canadá', 'ON': 'Canadá', 'QC': 'Canadá'
}

# Função para mapear estado -> macrorregião
def adicionar_coluna_macroregiao(df, coluna_estado='estado'):
    """
    Adiciona coluna 'macrorregiao' baseada no estado.
    """
    # Cria expressão CASE WHEN para todos os estados
    expr_macroregiao = None
    for estado, macroregiao in MAPA_MACROREGIAO.items():
        if expr_macroregiao is None:
            expr_macroregiao = when(col(coluna_estado) == estado, macroregiao)
        else:
            expr_macroregiao = expr_macroregiao.when(col(coluna_estado) == estado, macroregiao)
    
    # Default para estados não mapeados
    expr_macroregiao = expr_macroregiao.otherwise('Não classificado')
    
    return df.withColumn('macrorregiao', expr_macroregiao)

# ========== ATUALIZAR TABELA: gold_business_reviews_integrado ==========

print("\n1. Atualizando gold_business_reviews_integrado...")
table1 = f"{CATALOG}.{SCHEMA_GOLD}.gold_business_reviews_integrado"
df1 = spark.table(table1)

print(f"   Registros antes: {df1.count():,}")
print(f"   Colunas antes: {len(df1.columns)}")

# Adiciona macrorregião
df1_atualizado = adicionar_coluna_macroregiao(df1)

print(f"   Colunas depois: {len(df1_atualizado.columns)}")

# Salva tabela atualizada (com overwriteSchema para permitir adicionar coluna)
df1_atualizado.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table1)

print(f"   ✓ Tabela atualizada: {table1}")

# Mostra distribuição por macrorregião
print("\n   Distribuição por Macrorregião:")
df1_verificacao = spark.table(table1)
display(
    df1_verificacao
    .groupBy('macrorregiao')
    .count()
    .orderBy(col('count').desc())
)


In [0]:
# ========== TOP 10 RESTAURANTES POR SEGMENTO, ESTADO E ANO ==========

from pyspark.sql.functions import avg, count, desc, row_number, col, year
from pyspark.sql.window import Window

# Configuração
CATALOG = "workspace"
SCHEMA_GOLD = "yelp_gold"

print("Analisando os melhores restaurantes por segmento, localidade e ano...")
print("="*80)

# Carrega tabela integrada
df_integrado = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.gold_business_reviews_integrado")

print(f"Total de reviews na base: {df_integrado.count():,}")

# Extrai ano da data do review
df_integrado = df_integrado.withColumn('ano', year(col('date')))

# Agrupa por estabelecimento, categoria, estado, macrorregiao e ano
df_ranking = df_integrado.groupBy('estabelecimento', 'food_category', 'estado', 'macrorregiao', 'ano').agg(
    avg('stars').alias('media_stars'),
    count('review_id').alias('total_reviews')
)

# Filtra apenas estabelecimentos com pelo menos 50 reviews (para ter relevância estatística)
df_ranking = df_ranking.filter(col('total_reviews') >= 50)

print(f"\nEstabelecimentos com 50+ reviews: {df_ranking.count():,}")

# Cria window para rankear dentro de cada categoria + estado + ano
window_spec = Window.partitionBy('food_category', 'estado', 'ano').orderBy(desc('media_stars'), desc('total_reviews'))

# Adiciona ranking
df_ranking = df_ranking.withColumn('ranking', row_number().over(window_spec))

# Filtra apenas os top 10 de cada grupo
df_top10 = df_ranking.filter(col('ranking') <= 10)

print(f"\nTotal de registros no top 10: {df_top10.count():,}")

# Ordena o resultado final
df_top10_final = df_top10.orderBy('ano', 'food_category', 'estado', 'ranking')

print("\n✓ Análise concluída!")
print("="*80)

# Mostra resumo das combinações disponíveis
print("\nCOMBINAÇÕES DISPONÍVEIS (Categoria + Estado + Ano):")
df_combinacoes = df_top10_final.select('food_category', 'estado', 'ano').distinct().orderBy('ano', 'food_category', 'estado')
print(f"Total de combinações: {df_combinacoes.count()}")
print(f"\nAnos disponíveis: {df_top10_final.select('ano').distinct().count()}")
display(df_top10_final.select('ano').distinct().orderBy('ano'))

# Armazena o resultado
df_top10_cached = df_top10_final
print(f"\n✓ DataFrame com top 10 restaurantes criado: df_top10_cached")

In [0]:
# ========== EXEMPLOS DE TOP 10 POR CATEGORIA E ESTADO ==========

from pyspark.sql.functions import col

print("EXEMPLOS DE RANKINGS")
print("="*80)

# Exemplo 1: Top 10 RESTAURANTES na Pensilvânia (PA)
print("\n1. TOP 10 RESTAURANTES EM PENNSYLVANIA (PA):")
print("-"*80)
df_exemplo1 = df_top10_cached.filter(
    (col('food_category') == 'RESTAURANTE') & (col('estado') == 'PA')
).select(
    col('ranking').alias('Rank'),
    col('estabelecimento').alias('Restaurante'),
    col('media_stars').alias('Média Stars'),
    col('total_reviews').alias('Total Reviews')
)
display(df_exemplo1)

# Exemplo 2: Top 10 BARES na Flórida (FL)
print("\n2. TOP 10 BARES E BEBIDAS EM FLORIDA (FL):")
print("-"*80)
df_exemplo2 = df_top10_cached.filter(
    (col('food_category') == 'BAR E BEBIDA') & (col('estado') == 'FL')
).select(
    col('ranking').alias('Rank'),
    col('estabelecimento').alias('Bar'),
    col('media_stars').alias('Média Stars'),
    col('total_reviews').alias('Total Reviews')
)
display(df_exemplo2)

# Exemplo 3: Top 10 CAFES no Arizona (AZ)
print("\n3. TOP 10 CAFES EM ARIZONA (AZ):")
print("-"*80)
df_exemplo3 = df_top10_cached.filter(
    (col('food_category') == 'CAFE') & (col('estado') == 'AZ')
).select(
    col('ranking').alias('Rank'),
    col('estabelecimento').alias('Café'),
    col('media_stars').alias('Média Stars'),
    col('total_reviews').alias('Total Reviews')
)
display(df_exemplo3)



In [0]:
# ========== SALVAR TABELA TOP 10 COMO GOLD ==========

from pyspark.sql.functions import current_timestamp

# Configuração
CATALOG = "workspace"
SCHEMA_GOLD = "yelp_gold"

print("Salvando tabela de top 10 restaurantes na camada Gold...")
print("="*80)

# Nome da tabela
table_name = f"{CATALOG}.{SCHEMA_GOLD}.gold_top10_estabelecimentos"

# Adiciona metadados
df_top10_final_com_metadata = df_top10_cached.withColumn(
    'data_processamento',
    current_timestamp()
)

print(f"\nTotal de registros: {df_top10_final_com_metadata.count():,}")
print(f"Campos: {len(df_top10_final_com_metadata.columns)}")

# Salva a tabela (com overwriteSchema para permitir adicionar coluna ano)
df_top10_final_com_metadata.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

print(f"\n✓ Tabela salva com sucesso: {table_name}")
print("="*80)

# Verificação
print("\nVerificando tabela criada:")
df_verificacao = spark.table(table_name)
print(f"Total de registros: {df_verificacao.count():,}")

print("\nSchema:")
df_verificacao.printSchema()

# Mostra distribuição por ano
print("\nDistribuição por Ano:")
display(
    df_verificacao
    .groupBy('ano')
    .count()
    .orderBy('ano')
)

# Mostra distribuição por macrorregião e ano
print("\nDistribuição por Macrorregião e Ano (Top 5 anos):")
display(
    df_verificacao
    .groupBy('macrorregiao', 'ano')
    .count()
    .orderBy(col('ano').desc(), col('count').desc())
    .limit(20)
)

print("\nAmostra (5 registros):")
display(df_verificacao.limit(5))

# ========== ATUALIZAR TABELA: gold_top10_estabelecimentos ==========

print("\n2. Atualizando gold_top10_estabelecimentos...")
table2 = f"{CATALOG}.{SCHEMA_GOLD}.gold_top10_estabelecimentos"
df2 = spark.table(table2)

print(f"   Registros antes: {df2.count():,}")
print(f"   Colunas antes: {len(df2.columns)}")

# Adiciona macrorregião
df2_atualizado = adicionar_coluna_macroregiao(df2)

print(f"   Colunas depois: {len(df2_atualizado.columns)}")

# Salva tabela atualizada (com overwriteSchema para permitir adicionar coluna)
df2_atualizado.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table2)

print(f"   ✓ Tabela atualizada: {table2}")

# Mostra distribuição por macrorregião
print("\n   Distribuição por Macrorregião:")
df2_verificacao = spark.table(table2)
display(
    df2_verificacao
    .groupBy('macrorregiao')
    .count()
    .orderBy(col('count').desc())
)

print("\n" + "="*80)
print("✓ Coluna 'macrorregiao' adicionada com sucesso às tabelas!")
print("="*80)

# Mostra amostra com a nova coluna
print("\nAMOSTRA com macrorregião (gold_business_reviews_integrado):")
display(
    df1_verificacao.select(
        'estabelecimento', 'food_category', 'estado', 'macrorregiao', 'stars'
    ).limit(10)
)

